In [ ]:
# ============================================================
# STEP 1: IMPORT LIBRARIES AND SET RANDOM SEEDS
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger
)

# Reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception:
    print("Deterministic operations could not be fully enabled.")

print("TensorFlow version:", tf.__version__)

In [ ]:
# ============================================================
# RELIABLE GOOGLE DRIVE MOUNT
# ============================================================

import os
import shutil
from google.colab import drive

MOUNT_POINT = "/content/drive"

# Case 1: Drive is already properly mounted
if os.path.ismount(MOUNT_POINT):
    print("Google Drive is already mounted.")

# Case 2: The folder exists but is only a stale, unmounted folder
else:
    if os.path.exists(MOUNT_POINT):
        print("Removing stale /content/drive folder...")
        raise RuntimeError("Choose a fresh Colab runtime before mounting Drive.")

    # Create a clean mount point
    os.makedirs(MOUNT_POINT, exist_ok=True)

    # Mount Google Drive
    drive.mount(MOUNT_POINT)

# Final verification
if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is ready.")
    print("Mounted status:", os.path.ismount(MOUNT_POINT))
else:
    raise RuntimeError("Google Drive mounting failed.")

In [ ]:
import os

mri_folder = "/content/drive/MyDrive/ADNI_Project/MRI_Images/Processed_MRI_160"

files = sorted([f for f in os.listdir(mri_folder) if f.endswith(".npy")])

print("Total MRI files:", len(files))
print(files[:5])

# 4. Load final cognitive file for labels
final_cognitive = pd.read_csv(
    "/content/drive/MyDrive/ADNI_Project/Cognitive_Data/final_cognitive_702.csv"
)

print("Final cognitive shape:", final_cognitive.shape)

In [ ]:
# label_map = dict(zip(final_cognitive["RID"], final_cognitive["DIAGNOSIS"]))

# mri_records = []

# for file in files:
#     rid = int(re.search(r"RID_(.*?)_IMG", file).group(1))

#     if rid in label_map:
#         diagnosis = label_map[rid]
#         mri_records.append({
#             "RID": rid,
#             "MRI_FILE": file,
#             "MRI_PATH": os.path.join(mri_folder, file),
#             "DIAGNOSIS": diagnosis
#         })

# mri_dataset = pd.DataFrame(mri_records)

In [ ]:
# mri_dataset["LABEL"] = mri_dataset["DIAGNOSIS"].astype(int) - 1

# print("MRI dataset shape:", mri_dataset.shape)
# print(mri_dataset.head())
# print("\nClass distribution:")
# print(mri_dataset["LABEL"].value_counts().sort_index())

In [ ]:
# 6. Train/test split
# train_df, test_df = train_test_split(
#     mri_dataset,
#     test_size=0.20,
#     random_state=42,
#     stratify=mri_dataset["LABEL"]
#)

In [ ]:
# train_df, val_df = train_test_split(
#     train_df,
#     test_size=0.20,
#     random_state=42,
#     stratify=train_df["LABEL"]
# )

# print("Training:", train_df.shape)
# print("Validation:", val_df.shape)
# print("Testing:", test_df.shape)

# print("\nTraining class distribution:")
# print(train_df["LABEL"].value_counts().sort_index())

# print("\nValidation class distribution:")
# print(val_df["LABEL"].value_counts().sort_index())

# print("\nTesting class distribution:")
# print(test_df["LABEL"].value_counts().sort_index())

In [ ]:
# class_weights_array = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(train_df["LABEL"]),
#     y=train_df["LABEL"]
# )

# class_weights = dict(enumerate(class_weights_array))

# print("Class weights:", class_weights)



In [ ]:
# # 9. Three-slice MRI loader
# IMG_SIZE = 160
# BATCH_SIZE = 16

# SLICE_1 = 60
# SLICE_2 = 80
# SLICE_3 = 100

# def load_three_mri_slices(path, label):
#     path = path.numpy().decode("utf-8")
#     volume = np.load(path).astype(np.float32)

#     slice_1 = volume[:, :, SLICE_1]
#     slice_2 = volume[:, :, SLICE_2]
#     slice_3 = volume[:, :, SLICE_3]

#     image = np.stack([slice_1, slice_2, slice_3], axis=-1)

#     return image.astype(np.float32), label

# def tf_load_three_mri_slices(path, label):
#     image, label = tf.py_function(
#         load_three_mri_slices,
#         [path, label],
#         [tf.float32, tf.int64]
#     )

#     image.set_shape((IMG_SIZE, IMG_SIZE, 3))
#     label.set_shape([])

#     return image, label

In [ ]:
# 10. Create TensorFlow datasets
# train_dataset_3slice = tf.data.Dataset.from_tensor_slices(
#     (train_df["MRI_PATH"].values, train_df["LABEL"].values)
# )

# val_dataset_3slice = tf.data.Dataset.from_tensor_slices(
#     (val_df["MRI_PATH"].values, val_df["LABEL"].values)
# )

# test_dataset_3slice = tf.data.Dataset.from_tensor_slices(
#     (test_df["MRI_PATH"].values, test_df["LABEL"].values)
# )

# train_dataset_3slice = (
#     train_dataset_3slice
#     .map(tf_load_three_mri_slices, num_parallel_calls=tf.data.AUTOTUNE)
#     .shuffle(200)
#     .batch(BATCH_SIZE)
#     .prefetch(tf.data.AUTOTUNE)
# )

# val_dataset_3slice = (
#     val_dataset_3slice
#     .map(tf_load_three_mri_slices, num_parallel_calls=tf.data.AUTOTUNE)
#     .batch(BATCH_SIZE)
#     .prefetch(tf.data.AUTOTUNE)
# )

# test_dataset_3slice = (
#     test_dataset_3slice
#     .map(tf_load_three_mri_slices, num_parallel_calls=tf.data.AUTOTUNE)
#     .batch(BATCH_SIZE)
#     .prefetch(tf.data.AUTOTUNE)
# )

# print("\nThree-slice MRI datasets created successfully.")

In [ ]:
# 11. Check one batch
# for images, labels in train_dataset_3slice.take(1):
#     print("Image batch shape:", images.shape)
#     print("Label batch shape:", labels.shape)
#     print("Labels:", labels.numpy())
#     print("Image min:", tf.reduce_min(images).numpy())
#     print("Image max:", tf.reduce_max(images).numpy())



In [ ]:
# 12. Build 3-slice MobileNetV2 model
# base_model = MobileNetV2(
#     input_shape=(160, 160, 3),
#     include_top=False,
#     weights="imagenet"
# )
# # Keep pretrained MobileNetV2 frozen
# base_model.trainable = False

# mri_3slice_model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation="relu"),
#     layers.Dropout(0.4),
#     layers.Dense(3, activation="softmax")
# ])

# mri_3slice_model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# mri_3slice_model.summary()

In [ ]:
# 13. Train 3-slice MobileNetV2 model
# checkpoint_path = "/content/drive/MyDrive/ADNI_Project/MRI_Images/temporary_retrained_mri_model.keras"

# callbacks = [
#     EarlyStopping(
#         monitor="val_loss",
#         patience=5,
#         restore_best_weights=True
#     ),
#     ModelCheckpoint(
#         checkpoint_path,
#         monitor="val_loss",
#         save_best_only=True
#     )
# ]

# history_3slice = mri_3slice_model.fit(
#     train_dataset_3slice,
#     validation_data=val_dataset_3slice,
#     epochs=20,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

In [ ]:
# 14. Evaluate validation set
# val_loss, val_acc = mri_3slice_model.evaluate(val_dataset_3slice)

# print("\nValidation loss:", val_loss)
# print("Validation accuracy:", val_acc)

In [ ]:
# 15. Evaluate test set
#est_loss, test_acc = mri_3slice_model.evaluate(test_dataset_3slice)

#print("\nTest loss:", test_loss)
#print("Test accuracy:", test_acc)


In [ ]:
# 16. Confusion matrix and classification report
# y_true = []
# y_pred = []

# for images, labels in test_dataset_3slice:
#     preds = mri_3slice_model.predict(images, verbose=0)
#     y_true.extend(labels.numpy())
#     y_pred.extend(np.argmax(preds, axis=1))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_true, y_pred))

# print("\nClassification Report:")
# print(classification_report(
#     y_true,
#     y_pred,
#     target_names=["CN", "MCI", "AD"]
# ))

In [ ]:
# import os

# folder = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

# print("Files in MRI_Images folder:\n")

# for file in os.listdir(folder):
#     print(file)

In [ ]:
# import tensorflow as tf
# import numpy as np
# from sklearn.metrics import confusion_matrix, classification_report

# best_model_path = "/content/drive/MyDrive/ADNI_Project/MRI_Images/best_mri_3slice_mobilenet_model.keras"

# best_3slice_model = tf.keras.models.load_model(best_model_path)

# test_loss, test_acc = best_3slice_model.evaluate(test_dataset_3slice)

# print("Best checkpoint test loss:", test_loss)
# print("Best checkpoint test accuracy:", test_acc)

# y_true = []
# y_pred = []

# for images, labels in test_dataset_3slice:
#     preds = best_3slice_model.predict(images, verbose=0)
#     y_true.extend(labels.numpy())
#     y_pred.extend(np.argmax(preds, axis=1))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_true, y_pred))

# print("\nClassification Report:")
# print(classification_report(
#     y_true,
#     y_pred,
#     target_names=["CN", "MCI", "AD"]
# ))

In [ ]:
# ============================================================
# COPY EXISTING MRI MODEL WITHOUT RETRAINING
# ============================================================

# import os
# import shutil

# SOURCE_MODEL = (
#     "/content/drive/MyDrive/ADNI_Project/MRI_Images/"
#     "best_mri_3slice_mobilenet_model.keras"
# )

# DESTINATION_FOLDER = (
#     "/content/drive/MyDrive/ADNI_Project/Interface_Models"
# )

# DESTINATION_MODEL = os.path.join(
#     DESTINATION_FOLDER,
#     "best_mri_3slice_mobilenet_model.keras"
# )

# # Confirm that the already-trained model exists
# if not os.path.isfile(SOURCE_MODEL):
#     raise FileNotFoundError(
#         "The saved MRI model was not found:\n"
#         f"{SOURCE_MODEL}"
#     )

# os.makedirs(DESTINATION_FOLDER, exist_ok=True)

# # Avoid unnecessarily overwriting the interface copy
# if os.path.isfile(DESTINATION_MODEL):
#     print("MRI model is already present in Interface_Models.")
#     print("No copying or retraining was performed.")
# else:
#     shutil.copy2(SOURCE_MODEL, DESTINATION_MODEL)
#     print("Existing trained MRI model copied successfully.")
#     print("No retraining was performed.")

#print("Model location:", DESTINATION_MODEL)

In [ ]:
# ============================================================
# EVALUATE ALL SAVED MRI MODELS ON THE SAME TEST DATASET
# ============================================================

# import os
# import tensorflow as tf
# import pandas as pd

# MODEL_FOLDER = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

# model_files = [
#     "best_mri_cnn_model.keras",
#     "best_mri_mobilenet_model.keras",
#     "best_mri_3slice_mobilenet_model.keras",
#     "best_mri_3slice_mobilenet_finetuned.keras",
#     "best_mri_3slice_efficientnet_model.keras",
#     "temporary_retrained_mri_model.keras"
# ]

# results = []

# for model_file in model_files:

#     model_path = os.path.join(MODEL_FOLDER, model_file)

#     print("\n" + "=" * 70)
#     print("Evaluating:", model_file)
#     print("=" * 70)

#     if not os.path.isfile(model_path):
#         print("File not found.")
#         results.append({
#             "Model": model_file,
#             "Test Loss": None,
#             "Test Accuracy": None,
#             "Status": "File not found"
#         })
#         continue

#     try:
#         model = tf.keras.models.load_model(model_path)

#         test_loss, test_accuracy = model.evaluate(
#             test_dataset_3slice,
#             verbose=0
#         )

#         results.append({
#             "Model": model_file,
#             "Test Loss": test_loss,
#             "Test Accuracy": test_accuracy,
#             "Accuracy (%)": test_accuracy * 100,
#             "Status": "Evaluated successfully"
#         })

#         print(f"Test loss: {test_loss:.4f}")
#         print(f"Test accuracy: {test_accuracy * 100:.2f}%")

#     except Exception as error:

#         print("Could not evaluate this model.")
#         print("Reason:", error)

#         results.append({
#             "Model": model_file,
#             "Test Loss": None,
#             "Test Accuracy": None,
#             "Accuracy (%)": None,
#             "Status": str(error)
#         })

# results_df = pd.DataFrame(results)

# print("\n\nFINAL MODEL COMPARISON")
# display(results_df)

In [ ]:
# ============================================================
# STEP 2: CREATE MRI DATAFRAME WITH DIAGNOSIS LABELS
# ============================================================

label_map = dict(
    zip(
        final_cognitive["RID"].astype(int),
        final_cognitive["DIAGNOSIS"].astype(int)
    )
)

mri_records = []

for filename in files:

    match = re.search(r"RID_(.*?)_IMG", filename)

    if match is None:
        continue

    rid = int(match.group(1))

    if rid in label_map:

        diagnosis = label_map[rid]

        mri_records.append({
            "RID": rid,
            "MRI_FILE": filename,
            "MRI_PATH": os.path.join(mri_folder, filename),
            "DIAGNOSIS": diagnosis
        })

mri_dataset = pd.DataFrame(mri_records)

# Convert ADNI diagnosis labels:
# 1 = CN, 2 = MCI, 3 = AD
# into:
# 0 = CN, 1 = MCI, 2 = AD
mri_dataset["LABEL"] = (
    mri_dataset["DIAGNOSIS"].astype(int) - 1
)

mri_dataset = (
    mri_dataset
    .sort_values("RID")
    .reset_index(drop=True)
)

print("MRI dataset shape:", mri_dataset.shape)
print("\nFirst five records:")
display(mri_dataset.head())

print("\nClass distribution:")
print(
    mri_dataset["LABEL"]
    .value_counts()
    .sort_index()
)

assert len(mri_dataset) == 702, (
    f"Expected 702 records, but found {len(mri_dataset)}."
)

assert set(mri_dataset["LABEL"].unique()) == {0, 1, 2}, (
    "Unexpected diagnosis labels were found."
)

In [ ]:
# ============================================================
# STEP 3: STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First split: 80% development data, 20% independent test data
development_df, test_df = train_test_split(
    mri_dataset,
    test_size=0.20,
    random_state=SEED,
    stratify=mri_dataset["LABEL"]
)

# Second split: reserve 20% of development data for validation
train_df, val_df = train_test_split(
    development_df,
    test_size=0.20,
    random_state=SEED,
    stratify=development_df["LABEL"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\nTraining class distribution:")
print(train_df["LABEL"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["LABEL"].value_counts().sort_index())

print("\nTesting class distribution:")
print(test_df["LABEL"].value_counts().sort_index())

In [ ]:
# ============================================================
# STEP 4: SAVE THE DATA SPLITS FOR REPRODUCIBILITY
# ============================================================

RESULTS_FOLDER = (
    "/content/drive/MyDrive/ADNI_Project/"
    "Dissertation_Results/MRI_Results_Official"
)

os.makedirs(RESULTS_FOLDER, exist_ok=True)

train_df.to_csv(
    os.path.join(RESULTS_FOLDER, "MRI_training_split.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(RESULTS_FOLDER, "MRI_validation_split.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(RESULTS_FOLDER, "MRI_test_split.csv"),
    index=False
)

print("Data splits saved successfully.")
print("Results folder:", RESULTS_FOLDER)

In [ ]:
# ============================================================
# STEP 5: COMPUTE CLASS WEIGHTS
# ============================================================

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["LABEL"].values
)

class_weights = {
    class_index: float(weight)
    for class_index, weight in enumerate(class_weights_array)
}

print("Class weights:")
print(class_weights)

In [ ]:
# ============================================================
# STEP 6: THREE-SLICE MRI LOADER
# ============================================================

IMG_SIZE = 160
BATCH_SIZE = 16

SLICE_1 = 60
SLICE_2 = 80
SLICE_3 = 100

EXPECTED_VOLUME_SHAPE = (160, 160, 160)


def load_three_mri_slices(path, label):
    """
    Load one preprocessed MRI volume and construct a
    three-channel image using axial slices 60, 80 and 100.
    """

    path = path.numpy().decode("utf-8")

    volume = np.load(
        path,
        allow_pickle=False
    ).astype(np.float32)

    if volume.shape != EXPECTED_VOLUME_SHAPE:
        raise ValueError(
            f"Unexpected MRI shape {volume.shape} for {path}. "
            f"Expected {EXPECTED_VOLUME_SHAPE}."
        )

    slice_1 = volume[:, :, SLICE_1]
    slice_2 = volume[:, :, SLICE_2]
    slice_3 = volume[:, :, SLICE_3]

    # Form a pseudo-RGB image
    image = np.stack(
        [slice_1, slice_2, slice_3],
        axis=-1
    )

    # Ensure values remain between 0 and 1
    image = np.clip(image, 0.0, 1.0)

    # MobileNetV2 ImageNet preprocessing expects a 0–255 image
    # and transforms it to approximately -1 to 1.
    image = preprocess_input(image * 255.0)

    return image.astype(np.float32), np.int64(label)


def tf_load_three_mri_slices(path, label):

    image, label = tf.py_function(
        func=load_three_mri_slices,
        inp=[path, label],
        Tout=[tf.float32, tf.int64]
    )

    image.set_shape((IMG_SIZE, IMG_SIZE, 3))
    label.set_shape([])

    return image, label


print("Three-slice loader created successfully.")

In [ ]:
# ============================================================
# STEP 7: CREATE TENSORFLOW DATASETS
# ============================================================

train_dataset_3slice = tf.data.Dataset.from_tensor_slices(
    (
        train_df["MRI_PATH"].values,
        train_df["LABEL"].values
    )
)

val_dataset_3slice = tf.data.Dataset.from_tensor_slices(
    (
        val_df["MRI_PATH"].values,
        val_df["LABEL"].values
    )
)

test_dataset_3slice = tf.data.Dataset.from_tensor_slices(
    (
        test_df["MRI_PATH"].values,
        test_df["LABEL"].values
    )
)

train_dataset_3slice = (
    train_dataset_3slice
    .map(
        tf_load_three_mri_slices,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .shuffle(
        buffer_size=len(train_df),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset_3slice = (
    val_dataset_3slice
    .map(
        tf_load_three_mri_slices,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset_3slice = (
    test_dataset_3slice
    .map(
        tf_load_three_mri_slices,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("TensorFlow datasets created successfully.")

In [ ]:
# ============================================================
# STEP 8: VERIFY ONE TRAINING BATCH
# ============================================================

for images, labels in train_dataset_3slice.take(1):

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Labels:", labels.numpy())
    print("Minimum image value:", float(tf.reduce_min(images)))
    print("Maximum image value:", float(tf.reduce_max(images)))

In [ ]:
# ============================================================
# STEP 9: BUILD THREE-SLICE MOBILENETV2 MODEL
# ============================================================

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

mri_3slice_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.4),

    layers.Dense(
        3,
        activation="softmax"
    )
])

mri_3slice_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mri_3slice_model.summary()

In [ ]:
# ============================================================
# STEP 10: CALLBACKS AND OFFICIAL MODEL PATH
# ============================================================

OFFICIAL_MODEL_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_three_slice_mobilenet_model.keras"
)

TRAINING_LOG_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_training_log.csv"
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=OFFICIAL_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    CSVLogger(
        TRAINING_LOG_PATH,
        append=False
    )
]

print("Official model will be saved to:")
print(OFFICIAL_MODEL_PATH)

In [ ]:
# ============================================================
# STEP 11: TRAIN THE FINAL MRI MODEL
# ============================================================

history_3slice = mri_3slice_model.fit(
    train_dataset_3slice,
    validation_data=val_dataset_3slice,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed.")
print("Best model saved to:")
print(OFFICIAL_MODEL_PATH)

In [ ]:
# ============================================================
# STEP 12: SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history_3slice.history
)

history_df.index = (
    np.arange(1, len(history_df) + 1)
)

history_df.index.name = "Epoch"

HISTORY_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_training_history.csv"
)

history_df.to_csv(HISTORY_PATH)

print("Training history saved successfully:")
print(HISTORY_PATH)

display(history_df)

In [ ]:
# ============================================================
# STEP 13: TRAINING AND VALIDATION ACCURACY CURVE
# ============================================================

history_df = pd.read_csv(
    HISTORY_PATH,
    index_col="Epoch"
)

epochs = history_df.index.values

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(
    epochs,
    history_df["accuracy"],
    marker="o",
    linewidth=2.3,
    markersize=5,
    label="Training accuracy"
)

ax.plot(
    epochs,
    history_df["val_accuracy"],
    marker="s",
    linewidth=2.3,
    markersize=5,
    label="Validation accuracy"
)

ax.set_title(
    "Three-Slice MobileNetV2 Accuracy",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_xlabel(
    "Epoch",
    fontsize=11,
    fontweight="bold"
)

ax.set_ylabel(
    "Accuracy",
    fontsize=11,
    fontweight="bold"
)

ax.set_xticks(epochs)

ax.grid(
    linestyle="--",
    linewidth=0.8,
    alpha=0.35
)

ax.legend(
    fontsize=10,
    frameon=True
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

ACCURACY_FIGURE_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_accuracy_curve.png"
)

plt.savefig(
    ACCURACY_FIGURE_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Accuracy curve saved to:")
print(ACCURACY_FIGURE_PATH)

In [ ]:
# ============================================================
# STEP 14: TRAINING AND VALIDATION LOSS CURVE
# ============================================================

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(
    epochs,
    history_df["loss"],
    marker="o",
    linewidth=2.3,
    markersize=5,
    label="Training loss"
)

ax.plot(
    epochs,
    history_df["val_loss"],
    marker="s",
    linewidth=2.3,
    markersize=5,
    label="Validation loss"
)

ax.set_title(
    "Three-Slice MobileNetV2 Loss",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_xlabel(
    "Epoch",
    fontsize=11,
    fontweight="bold"
)

ax.set_ylabel(
    "Loss",
    fontsize=11,
    fontweight="bold"
)

ax.set_xticks(epochs)

ax.grid(
    linestyle="--",
    linewidth=0.8,
    alpha=0.35
)

ax.legend(
    fontsize=10,
    frameon=True
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

LOSS_FIGURE_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_loss_curve.png"
)

plt.savefig(
    LOSS_FIGURE_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Loss curve saved to:")
print(LOSS_FIGURE_PATH)

In [ ]:
# ============================================================
# STEP 15: LOAD AND EVALUATE THE SAVED BEST MODEL
# ============================================================

official_mri_model = tf.keras.models.load_model(
    OFFICIAL_MODEL_PATH
)

test_loss, test_accuracy = official_mri_model.evaluate(
    test_dataset_3slice,
    verbose=1
)

print("\nOfficial MRI model results")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# ============================================================
# STEP 16: GENERATE TEST PREDICTIONS
# ============================================================

y_true = []
y_probabilities = []

for images, labels in test_dataset_3slice:

    batch_probabilities = official_mri_model.predict(
        images,
        verbose=0
    )

    y_true.extend(labels.numpy())
    y_probabilities.extend(batch_probabilities)

y_true = np.asarray(y_true)
y_probabilities = np.asarray(y_probabilities)

y_pred = np.argmax(
    y_probabilities,
    axis=1
)

print("Number of test predictions:", len(y_pred))
print(
    "Calculated test accuracy:",
    f"{accuracy_score(y_true, y_pred) * 100:.2f}%"
)

In [ ]:
# ============================================================
# STEP 17: OFFICIAL MRI CONFUSION MATRIX
# ============================================================

cmatrix = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

class_names = ["CN", "MCI", "AD"]

fig, ax = plt.subplots(figsize=(6, 5))

image = ax.imshow(
    cmatrix,
    cmap="GnBu",
    interpolation="nearest"
)

colour_bar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

colour_bar.set_label(
    "Count",
    fontsize=10
)

ax.set_xticks(np.arange(3))
ax.set_yticks(np.arange(3))

ax.set_xticklabels(
    class_names,
    fontsize=10
)

ax.set_yticklabels(
    class_names,
    fontsize=10
)

ax.set_xlabel(
    "Predicted Diagnosis",
    fontsize=11,
    fontweight="bold",
    labelpad=10
)

ax.set_ylabel(
    "Actual Diagnosis",
    fontsize=11,
    fontweight="bold",
    labelpad=10
)

ax.set_title(
    "Three-Slice MobileNetV2 Confusion Matrix",
    fontsize=14,
    fontweight="bold",
    pad=14
)

threshold = cmatrix.max() / 2

for row in range(3):
    for column in range(3):

        value = cmatrix[row, column]

        ax.text(
            column,
            row,
            str(value),
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold",
            color=(
                "white"
                if value > threshold
                else "#183c3c"
            )
        )

ax.set_xticks(
    np.arange(-0.5, 3, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, 3, 1),
    minor=True
)

ax.grid(
    which="minor",
    linewidth=1.2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

plt.tight_layout()

CONFUSION_MATRIX_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_confusion_matrix.png"
)

plt.savefig(
    CONFUSION_MATRIX_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Confusion matrix:")
print(cmatrix)

print("\nConfusion matrix saved to:")
print(CONFUSION_MATRIX_PATH)

In [ ]:
# ============================================================
# STEP 18: OFFICIAL MRI CLASSIFICATION REPORT
# ============================================================

report_dictionary = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["CN", "MCI", "AD"],
    output_dict=True,
    zero_division=0
)

classification_report_df = (
    pd.DataFrame(report_dictionary)
    .transpose()
)

CLASSIFICATION_REPORT_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_classification_report.csv"
)

classification_report_df.to_csv(
    CLASSIFICATION_REPORT_PATH
)

display(
    classification_report_df.round(3)
)

print("Classification report saved to:")
print(CLASSIFICATION_REPORT_PATH)

In [ ]:
# ============================================================
# STEP 19: SAVE FINAL MRI RESULT SUMMARY
# ============================================================

summary_df = pd.DataFrame({
    "Metric": [
        "Test loss",
        "Test accuracy",
        "Number of training participants",
        "Number of validation participants",
        "Number of test participants",
        "Batch size",
        "Maximum epochs",
        "Learning rate",
        "Best model file"
    ],
    "Value": [
        round(float(test_loss), 6),
        round(float(test_accuracy), 6),
        len(train_df),
        len(val_df),
        len(test_df),
        BATCH_SIZE,
        20,
        0.0001,
        OFFICIAL_MODEL_PATH
    ]
})

SUMMARY_PATH = os.path.join(
    RESULTS_FOLDER,
    "official_mri_results_summary.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

display(summary_df)

print("\nAll official MRI results have been saved in:")
print(RESULTS_FOLDER)